# 03 · Escudos de los 20 equipos y color primario

Descarga el escudo de cada equipo, lo normaliza para que los 20 se vean como un juego y extrae su color
dominante.

**Fuentes, por este orden:**

1. La propiedad P154 (logotipo) de Wikidata, que apunta a un fichero de Wikimedia Commons con licencia libre.
   Solo la tienen 3 de los 20 clubes: Barcelona, Celta y Getafe.
2. Para el resto, la web oficial del club. La url de cada escudo se eligió a mano mirando la página, y
   está escrita abajo.

**Normalización.** Se recorta el margen transparente del original, se escala para que quepa en un cuadro de
112 × 112 px sin deformarlo y se centra en un lienzo transparente de 128 × 128 px. Todos tienen así el mismo
tamaño y el mismo margen interior: 8 px en el lado que más ocupa.

**Color primario.** Es el color más frecuente de los píxeles opacos del escudo, sin contar el blanco ni el
negro (ver la regla en el apartado 4).

In [1]:
# Instalamos las librerías del cuaderno; resvg-py convierte SVG a PNG sin dependencias del sistema
%pip install -q pandas requests pillow resvg-py

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos io para abrir imágenes desde memoria
import io
# Importamos re para limpiar el HTML del autor
import re
# Importamos datetime para la fecha de descarga
import datetime
# Importamos requests para descargar
import requests
# Importamos pandas para la tabla de escudos
import pandas as pd
# Importamos Pillow para tratar las imágenes
from PIL import Image, ImageDraw
# Importamos resvg_py para convertir los SVG
import resvg_py

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Carpeta de destino de los escudos
CARPETA = "images/escudos"
# La creamos si no existe
os.makedirs(CARPETA, exist_ok=True)
# Las webs de los clubes no responden a clientes sin navegador: nos identificamos como uno
CABECERAS_WEB = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130 Safari/537.36"}
# Identificación para la API de Wikimedia, como piden sus normas de uso
CABECERAS_WIKI = {"User-Agent": "scouting-laliga-2526/1.0 (proyecto de datos; python-requests)"}

## 1. Origen de cada escudo

In [3]:
# (equipo del modelo, fuente, fichero de Commons o url de la web oficial)
ORIGEN = [
    ("Alaves", "web", "https://cms.deportivoalaves.com/uploads/da_escudo_67a7d46780.png"),
    ("Athletic Club", "web", "https://cdn.athletic-club.eus/imagenes/escudos/Athletic-club-400x400.png"),
    ("Atletico Madrid", "web", "https://www.atleticodemadrid.com/images/EscudoATM.svg"),
    ("Barcelona", "wikidata", "File:Football Club Barcelona Color Update.png"),
    ("Celta Vigo", "wikidata", "File:Escudo RC Celta de Vigo.svg"),
    ("Elche", "web", "https://statics-maker.llt-services.com/elc/images/2022/11/14/xlarge/bcf670fa73b008488a406954731afe81.png"),
    ("Espanyol", "web", "https://www.rcdespanyol.com/assets/images/logo/top_emblem.svg"),
    ("Getafe", "wikidata", "File:Getafe CF Logo.png"),
    ("Girona", "web", "https://statics-maker.llt-services.com/gir/images/2024/02/05/original/ec06d70b-44be-4189-a646-cf5bd991bfd2-112.png"),
    ("Levante", "web", "https://statics-maker.llt-services.com/lev/images/2023/01/24/xlarge/0c81dc1cd02f4b7ed22d845cc88fb94d.png"),
    ("Mallorca", "web", "https://statics-maker.llt-services.com/master-football/football-images/66b3ee07d6c00f8367585b1c94289346.png"),
    ("Osasuna", "web", "https://www.osasuna.es/images/teams/osasuna.png"),
    ("Rayo Vallecano", "web", "https://statics-maker.llt-services.com/ray/images/2023/01/05/xlarge/a4b97252531a547f7bdf1b272973e85d.png"),
    ("Real Betis", "web", "https://www.realbetisbalompie.es/media/img/graphics/new_logos/logo_shield.svg"),
    ("Real Madrid", "web", "https://assets.realmadrid.com/is/image/realmadrid/3kq9cckrnlogidldtdie2fkbl?$Mobile$&fit=wrap&wid=144&hei=144"),
    ("Real Oviedo", "web", "https://statics-maker.llt-services.com/ovi/images/2025/05/16/large/bb23a74b-1640-427c-9763-22db78a845e7-841.png"),
    ("Real Sociedad", "web", "https://www.realsociedad.eus/Content/img/Escudo-head.png"),
    ("Sevilla", "web", "https://www.sevillafc.es/favicon/android-chrome-192x192.png"),
    ("Valencia", "web", "https://www.valenciacf.com/svg/escudo.svg"),
    ("Villarreal", "web", "https://panel.villarrealcf.es/wp-content/uploads/2026/06/escudo_300x300_new.png.webp"),
]

## 2. Descarga

In [4]:
# Fecha de hoy, para la tabla
HOY = datetime.date.today().isoformat()


# Descarga un escudo y devuelve (imagen RGBA, url real, licencia)
def descargar(fuente, ref):
    # Escudo de Wikimedia Commons
    if fuente == "wikidata":
        # Pedimos la imagen como PNG de 512 px de ancho, su url original y su licencia
        res = requests.get("https://commons.wikimedia.org/w/api.php", headers=CABECERAS_WIKI, timeout=30, params={
            "action": "query", "titles": ref, "prop": "imageinfo", "iiprop": "url|extmetadata",
            "iiurlwidth": 512, "format": "json", "formatversion": "2"}).json()
        # Información de la imagen
        ii = res["query"]["pages"][0]["imageinfo"][0]
        # Descargamos la versión PNG
        datos = requests.get(ii.get("thumburl") or ii["url"], headers=CABECERAS_WIKI, timeout=30).content
        # Licencia del fichero
        licencia = ii["extmetadata"]["LicenseShortName"]["value"]
        # Autor del fichero, sin etiquetas HTML
        autor = " ".join(re.sub(r"<[^>]+>", "", ii["extmetadata"].get("Artist", {}).get("value", "")).split()) or None
        # Devolvemos la imagen, la página del fichero, la licencia y el autor
        return Image.open(io.BytesIO(datos)).convert("RGBA"), ii["descriptionurl"], licencia, autor
    # Escudo de la web oficial
    r = requests.get(ref, headers=CABECERAS_WEB, timeout=30)
    # Si falla, paramos
    r.raise_for_status()
    # Si es un SVG, lo pasamos a PNG de 512 px de ancho
    if ref.split("?")[0].endswith(".svg"):
        # Conversión con resvg
        datos = bytes(resvg_py.svg_to_bytes(svg_string=r.text, width=512))
    else:
        # El resto ya es una imagen rasterizada
        datos = r.content
    # Las imágenes de las webs oficiales no tienen licencia libre
    return Image.open(io.BytesIO(datos)).convert("RGBA"), ref, "marca registrada del club; descargado de su web oficial", None


# Descargamos los 20
originales = {}
# Filas de la tabla de escudos
filas = []
for equipo, fuente, ref in ORIGEN:
    # Descarga
    im, url, licencia, autor = descargar(fuente, ref)
    # Guardamos el original en memoria
    originales[equipo] = im
    # Fila de la tabla
    filas.append({"equipo": equipo, "fuente": fuente, "url_origen": url, "licencia": licencia, "autor": autor,
                  "fecha_descarga": HOY, "ancho_original": im.width, "alto_original": im.height})
    # Informamos
    print(f"{equipo:<16} {fuente:<9} {im.width:>5} x {im.height:<5} {licencia}")

Alaves           web         170 x 168   marca registrada del club; descargado de su web oficial
Athletic Club    web         400 x 400   marca registrada del club; descargado de su web oficial


Atletico Madrid  web         512 x 657   marca registrada del club; descargado de su web oficial


Barcelona        wikidata    960 x 972   CC0


Celta Vigo       wikidata    960 x 1305  CC BY-SA 3.0
Elche            web         740 x 740   marca registrada del club; descargado de su web oficial


Espanyol         web         512 x 512   marca registrada del club; descargado de su web oficial


Getafe           wikidata    354 x 354   CC BY-SA 4.0
Girona           web         740 x 740   marca registrada del club; descargado de su web oficial


... (3 outputs left out)

Real Madrid      web         144 x 144   marca registrada del club; descargado de su web oficial
Real Oviedo      web        1200 x 1200  marca registrada del club; descargado de su web oficial


Real Sociedad    web          89 x 100   marca registrada del club; descargado de su web oficial


Sevilla          web         192 x 192   marca registrada del club; descargado de su web oficial


Valencia         web         512 x 668   marca registrada del club; descargado de su web oficial
Villarreal       web         300 x 300   marca registrada del club; descargado de su web oficial


## 3. Normalización a 128 × 128 con el mismo margen

In [5]:
# Tamaño final y cuadro útil
LADO, CAJA = 128, 112


# Recorta el margen transparente, escala al cuadro útil y centra en el lienzo
def normalizar(im):
    # Caja de los píxeles con algo de opacidad
    caja = im.getchannel("A").point(lambda a: 255 if a > 10 else 0).getbbox()
    # Recortamos el margen transparente
    im = im.crop(caja)
    # Escala para que el lado mayor mida 112 px
    escala = CAJA / max(im.size)
    # Nuevo tamaño, sin deformar
    nuevo = (max(1, round(im.width * escala)), max(1, round(im.height * escala)))
    # Redimensionamos con buen filtro
    im = im.resize(nuevo, Image.LANCZOS)
    # Lienzo transparente
    lienzo = Image.new("RGBA", (LADO, LADO), (0, 0, 0, 0))
    # Pegamos centrado
    lienzo.alpha_composite(im, ((LADO - im.width) // 2, (LADO - im.height) // 2))
    # Devolvemos el escudo normalizado
    return lienzo


# Nombre de fichero de cada equipo: el mismo que usa dim_equipo
dim_equipo = pd.read_csv("data/modelo/dim_equipo.csv")
# Correspondencia equipo → fichero
FICHERO = dict(zip(dim_equipo["equipo"], dim_equipo["escudo"]))
# Recorremos los escudos
for f in filas:
    # Normalizamos
    im = normalizar(originales[f["equipo"]])
    # Ruta de destino
    ruta = os.path.join(CARPETA, FICHERO[f["equipo"]])
    # Guardamos optimizado
    im.save(ruta, optimize=True)
    # Anotamos fichero, escala y peso
    f["fichero"] = FICHERO[f["equipo"]]
    f["kb"] = round(os.path.getsize(ruta) / 1024, 1)
    # Factor de escala aplicado al escudo recortado (más de 1 = se ha ampliado)
    caja = originales[f["equipo"]].getchannel("A").point(lambda a: 255 if a > 10 else 0).getbbox()
    f["escala"] = round(CAJA / max(caja[2] - caja[0], caja[3] - caja[1]), 2)

# Comprobaciones: tamaño, transparencia y peso
for f in filas:
    # Abrimos el resultado
    im = Image.open(os.path.join(CARPETA, f["fichero"]))
    # 128 x 128 con canal alfa y por debajo de 50 KB
    assert im.size == (128, 128) and im.mode == "RGBA" and f["kb"] < 50, f
# Informamos
print("20 escudos de 128 x 128, con transparencia y por debajo de 50 KB")
print("ampliados (escala > 1):", [(f["equipo"], f["escala"]) for f in filas if f["escala"] > 1])

20 escudos de 128 x 128, con transparencia y por debajo de 50 KB
ampliados (escala > 1): [('Real Sociedad', 1.12)]


## 4. Color primario

Regla: de los píxeles opacos (alfa ≥ 200) del escudo original, se descartan los blancos (los tres canales
≥ 220) y los negros (los tres canales ≤ 45). Los restantes se agrupan en 8 colores con el algoritmo de corte
mediano de Pillow, y el color primario es el grupo con más píxeles.

In [6]:
# Calcula el color dominante de un escudo, sin blanco ni negro
def color_primario(im):
    # Reducimos para que el cálculo sea rápido
    im = im.copy()
    im.thumbnail((256, 256))
    # Píxeles RGBA
    px = list(im.get_flattened_data()) if hasattr(im, "get_flattened_data") else list(im.getdata())
    # Píxeles opacos que no son ni blancos ni negros
    validos = [p[:3] for p in px if p[3] >= 200 and not min(p[:3]) >= 220 and not max(p[:3]) <= 45]
    # Imagen de una fila con esos píxeles
    tira = Image.new("RGB", (len(validos), 1))
    tira.putdata(validos)
    # Agrupamos en 8 colores
    q = tira.quantize(colors=8, method=Image.Quantize.MEDIANCUT)
    # Recuento de píxeles por color, de más a menos
    cuenta = sorted(q.getcolors(), reverse=True)
    # Paleta de la imagen cuantizada
    paleta = q.getpalette()
    # Color del grupo más numeroso
    r, g, b = paleta[cuenta[0][1] * 3: cuenta[0][1] * 3 + 3]
    # Peso de ese color entre los píxeles válidos
    peso = cuenta[0][0] / len(validos) * 100
    # Devolvemos el color en hexadecimal y su peso
    return f"#{r:02x}{g:02x}{b:02x}", round(peso, 1)


# Calculamos el color de cada escudo
for f in filas:
    # Color y peso
    f["color_primario"], f["peso_color_pct"] = color_primario(originales[f["equipo"]])

## 5. Guardado y hoja de control

In [7]:
# Tabla de escudos con la clave del modelo
escudos = pd.DataFrame(filas).merge(dim_equipo[["id_equipo", "equipo"]], on="equipo")
# Columnas en orden
escudos = escudos[["id_equipo", "equipo", "fichero", "fuente", "url_origen", "licencia", "autor", "fecha_descarga",
                   "ancho_original", "alto_original", "escala", "kb", "color_primario", "peso_color_pct"]]
# Guardamos
escudos.sort_values("id_equipo").to_csv("data/escudos.csv", index=False)
# Mostramos los colores
print(escudos[["equipo", "fuente", "color_primario", "peso_color_pct", "escala", "kb"]].to_string(index=False))

# Hoja de control: cada escudo sobre gris, con una muestra de su color debajo
hoja = Image.new("RGBA", (5 * 150, 4 * 190), (225, 225, 225, 255))
# Lápiz para el texto y las muestras
d = ImageDraw.Draw(hoja)
# Recorremos los 20 en orden de clave
for n, f in enumerate(escudos.sort_values("id_equipo").itertuples()):
    # Posición de la celda
    x, y = (n % 5) * 150 + 11, (n // 5) * 190 + 8
    # Escudo normalizado
    hoja.alpha_composite(Image.open(os.path.join(CARPETA, f.fichero)), (x, y))
    # Muestra de color
    d.rectangle([x, y + 132, x + 128, y + 150], fill=f.color_primario)
    # Nombre y color
    d.text((x, y + 154), f"{f.equipo} {f.color_primario}", fill=(0, 0, 0, 255))
# Guardamos la hoja fuera de la carpeta de escudos
hoja.save("images/escudos_control.png")

         equipo   fuente color_primario  peso_color_pct  escala   kb
         Alaves      web        #0033a0            76.4    0.67 23.1
  Athletic Club      web        #e11f1c            47.8    0.28 12.9
Atletico Madrid      web        #282a6f            46.6    0.17  8.6
      Barcelona wikidata        #deb500            36.5    0.12 12.5
     Celta Vigo wikidata        #ff0000            51.7    0.09  7.4
          Elche      web        #e0cf9c            28.6    0.22 14.7
       Espanyol      web        #e01517            31.7    0.22 14.4
         Getafe wikidata        #0062a5            25.8    0.33 22.3
         Girona      web        #d02030            55.1    0.19 21.9
        Levante      web        #fcc60c            24.6    0.14 17.2
       Mallorca      web        #ffe600            33.8    0.16 16.2
        Osasuna      web        #b89e14            33.1    0.94 14.9
 Rayo Vallecano      web        #c0ae33            53.0    0.70 17.8
     Real Betis      web        #c